In [1]:
import pandas as pd
import numpy as np
import os

def deep_profile_master_table(file_path="Final_Integrated_Credit_Master_v2_Complete.csv"):
    print(f"🔬 [Deep Dive Audit] 통합 마스터 테이블 정밀 프로파일링 시작...")
    print("-" * 90)
    
    if not os.path.exists(file_path):
        print(f"❌ {file_path} 파일을 찾을 수 없습니다.")
        return
        
    df = pd.read_csv(file_path, dtype={'COMPANY_ID': str})
    total_companies = len(df)
    
    print(f"▶️ 총 기업 수(Rows): {total_companies:,}개")
    print(f"▶️ 통합된 피처 수(Cols): {df.shape[1]}개\n")

    # 1. 컬럼별 정밀 통계 프로파일링
    profile_data = []
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_ratio = (null_count / total_companies) * 100
        unique_count = df[col].nunique()
        
        # 수치형 데이터인 경우 최소, 최대, 평균 계산
        if pd.api.types.is_numeric_dtype(df[col]):
            min_val = round(df[col].min(), 2) if not pd.isna(df[col].min()) else "NaN"
            max_val = round(df[col].max(), 2) if not pd.isna(df[col].max()) else "NaN"
            mean_val = round(df[col].mean(), 2) if not pd.isna(df[col].mean()) else "NaN"
            zero_count = (df[col] == 0).sum()
        else:
            min_val, max_val, mean_val, zero_count = "-", "-", "-", "-"
            
        profile_data.append({
            'Feature': col,
            'Dtype': str(df[col].dtype),
            'Nulls': f"{null_count} ({null_ratio:.1f}%)",
            'Zeros': zero_count,
            'Unique': unique_count,
            'Min': min_val,
            'Mean': mean_val,
            'Max': max_val
        })
        
    profile_df = pd.DataFrame(profile_data)
    print("📊 [Feature Profiling Report]")
    print(profile_df.to_string(index=False))
    
    # 2. 핵심 타겟(Core Cohort) 교집합 분석
    print("\n" + "=" * 90)
    print("🎯 [Core Cohort Analysis] 모델링 유효 모수 분석")
    
    # 조건 1: 재무 데이터(유동비율 등)가 존재하는 기업
    if 'LIQUIDITY_RATIO' in df.columns:
        has_fin = df['LIQUIDITY_RATIO'].notnull()
        fin_count = has_fin.sum()
        print(f" ✔️ 재무 데이터 보유 기업: {fin_count:,}개 ({fin_count/total_companies*100:.1f}%)")
    else:
        has_fin = pd.Series(True, index=df.index)
        
    # 조건 2: 동적 활동 데이터(매출채권 내역 OR BNPL 신청 내역)가 1건이라도 있는 기업
    has_dyn = (df.get('receivable_Partner_Count', 0) > 0) | (df.get('BNPL_Req_Count', 0) > 0)
    dyn_count = has_dyn.sum()
    print(f" ✔️ BNPL/채권 활동 내역 보유 기업: {dyn_count:,}개 ({dyn_count/total_companies*100:.1f}%)")
    
    # 조건 1 & 2 교집합
    core_cohort = df[has_fin & has_dyn]
    core_count = len(core_cohort)
    print(f" 🚀 [진성 고객] 재무 + 활동 데이터 모두 보유: {core_count:,}개 ({core_count/total_companies*100:.1f}%)")
    print("=" * 90)

# 즉시 실행
deep_profile_master_table()

🔬 [Deep Dive Audit] 통합 마스터 테이블 정밀 프로파일링 시작...
------------------------------------------------------------------------------------------
▶️ 총 기업 수(Rows): 1,514개
▶️ 통합된 피처 수(Cols): 16개

📊 [Feature Profiling Report]
                 Feature   Dtype        Nulls Zeros  Unique      Min          Mean             Max
              COMPANY_ID  object     0 (0.0%)     -    1514        -             -               -
    receivable_Total_Amt float64 1217 (80.4%)   140     153      0.0  2044523170.7  180013947834.0
receivable_Partner_Count float64 1217 (80.4%)   121      43      0.0         42.57          4129.0
Receivable_Concentration float64 1217 (80.4%)   140     111      0.0          0.25             1.0
          BNPL_Req_Count float64 1217 (80.4%)   200      12      0.0          1.03           110.0
            BNPL_Avg_Amt float64 1217 (80.4%)   203      84      0.0   37571126.09    1647724455.0
       BNPL_Success_Rate float64 1217 (80.4%)   281      11      0.0          0.04           

In [2]:
import pandas as pd
import numpy as np
import os

def build_schema_mapped_master(root_path='.'):
    print("🏦 [Schema Mapped Engine] 스키마 명세 기반 핵심 재무 지표 통합 가동...")
    print("-" * 90)
    
    # 1. Base (개황) + Dynamic (BNPL/채권) 로드 (기존과 동일)
    master = pd.DataFrame()
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_OVERVIEW.csv' in files and master.empty:
            df_base = pd.read_csv(os.path.join(root, 'COMPANY_OVERVIEW.csv'), dtype={'COMPANY_ID': str})
            df_base['COMPANY_ID'] = df_base['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            master = df_base.drop_duplicates(subset=['COMPANY_ID'], keep='last')[['COMPANY_ID']]

    dynamic_file = 'Dynamic_Feature_Mart_v1.csv'
    if os.path.exists(dynamic_file):
        df_dyn = pd.read_csv(dynamic_file, dtype={'COMPANY_ID': str})
        df_dyn['COMPANY_ID'] = df_dyn['COMPANY_ID'].str.zfill(10)
        master = pd.merge(master, df_dyn, on='COMPANY_ID', how='left')

    # =========================================================
    # 2. [스키마 기반 매핑] COMPANY_FINANCIAL_RATIO (6대 지표)
    # =========================================================
    print("▶️ [1/2] 재무비율 스키마 정밀 매핑 중 (FR_VAL)...")
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_FINANCIAL_RATIO.csv' in files:
            df_fin = pd.read_csv(os.path.join(root, 'COMPANY_FINANCIAL_RATIO.csv'), dtype={'COMPANY_ID': str})
            df_fin['COMPANY_ID'] = df_fin['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_fin = df_fin.sort_values(by=['COMPANY_ID', 'FR_ACCT_DT']).drop_duplicates(subset=['COMPANY_ID'], keep='last')
            
            # 스키마 명세서에 따른 정확한 매핑 딕셔너리
            schema_mapping = {
                'FR_VAL15': 'LIQUIDITY_RATIO',         # 유동비율
                'FR_VAL7':  'DEBT_RATIO',              # 부채비율
                'FR_VAL8':  'INTEREST_COVERAGE_RATIO', # 이자보상배율
                'FR_VAL4':  'OPERATING_MARGIN',        # 영업이익률
                'FR_VAL2':  'SALES_GROWTH_RATE',       # 매출액증가율
                'FR_VAL9':  'BORROWING_DEPENDENCE'     # 차입금의존도
            }
            
            df_fin_mapped = df_fin.rename(columns=schema_mapping)
            target_cols = ['COMPANY_ID'] + list(schema_mapping.values())
            target_cols = [c for c in target_cols if c in df_fin_mapped.columns]
            master = pd.merge(master, df_fin_mapped[target_cols], on='COMPANY_ID', how='left')

    # =========================================================
    # 3. [스키마 기반 연산] COMPANY_FINANCIAL_STATEMENT (현금비율)
    # =========================================================
    print("▶️ [2/2] 대차대조표 스키마 연산 중 (현금비율 계산)...")
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_FINANCIAL_STATEMENT.csv' in files:
            df_state = pd.read_csv(os.path.join(root, 'COMPANY_FINANCIAL_STATEMENT.csv'), dtype={'COMPANY_ID': str})
            df_state['COMPANY_ID'] = df_state['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_state = df_state.sort_values(by=['COMPANY_ID', 'FS_ACCT_DT']).drop_duplicates(subset=['COMPANY_ID'], keep='last')
            
            # 컬럼명이 대문자일 수 있으므로 소문자로 강제 변환 후 매핑 시도
            df_state.columns = df_state.columns.str.lower()
            
            # 현금비율 = (현금 및 현금성 자산(fs_val4) + 외화현금(fs_val6)) / 유동부채(fs_val238) * 100
            if set(['fs_val4', 'fs_val6', 'fs_val238']).issubset(df_state.columns):
                # 0으로 나누는 에러 방지
                df_state['fs_val238'] = df_state['fs_val238'].replace(0, np.nan)
                df_state['CASH_RATIO'] = ((df_state['fs_val4'].fillna(0) + df_state['fs_val6'].fillna(0)) / df_state['fs_val238']) * 100
                
                # COMPANY_ID 컬럼을 다시 대문자로 변경하여 병합
                df_state = df_state.rename(columns={'company_id': 'COMPANY_ID'})
                master = pd.merge(master, df_state[['COMPANY_ID', 'CASH_RATIO']], on='COMPANY_ID', how='left')
                print("  └─ ✅ 현금비율(CASH_RATIO) 산출 및 결합 성공!")
            else:
                print("  └─ ⚠️ 필요 컬럼(fs_val4, 6, 238)이 누락되어 현금비율 계산 불가")

    # 4. 정성/신뢰도 지표 결합 (이전과 동일)
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_EMPLOYEE_STATISTICS.csv' in files:
            df_emp = pd.read_csv(os.path.join(root, 'COMPANY_EMPLOYEE_STATISTICS.csv'), dtype={'COMPANY_ID': str})
            df_emp['COMPANY_ID'] = df_emp['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_emp_latest = df_emp.sort_values(by=['COMPANY_ID', 'STANDARD_DATE']).drop_duplicates(subset=['COMPANY_ID'], keep='last')
            master = pd.merge(master, df_emp_latest[['COMPANY_ID', 'EMPLOYEE_COUNT']], on='COMPANY_ID', how='left')
            
        if 'COMPANY_REPRESENTATIVE_HISTORY.csv' in files:
            df_rep = pd.read_csv(os.path.join(root, 'COMPANY_REPRESENTATIVE_HISTORY.csv'), dtype={'COMPANY_ID': str})
            df_rep['COMPANY_ID'] = df_rep['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            rep_agg = df_rep.groupby('COMPANY_ID').size().reset_index(name='REP_CHANGE_COUNT')
            master = pd.merge(master, rep_agg, on='COMPANY_ID', how='left')

        if 'COMPANY_LINKS.csv' in files:
            df_link = pd.read_csv(os.path.join(root, 'COMPANY_LINKS.csv'), dtype={'COMPANY_ID': str})
            df_link['COMPANY_ID'] = df_link['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            link_agg = df_link.groupby('COMPANY_ID').agg(LINKED_PARTNERS=('ANOTHER_COMPANY_ID', 'nunique')).reset_index()
            master = pd.merge(master, link_agg, on='COMPANY_ID', how='left')
            
        if 'PAY_BNPL_COMMENTS.csv' in files:
            df_comm = pd.read_csv(os.path.join(root, 'PAY_BNPL_COMMENTS.csv'), dtype={'COMPANY_ID': str})
            df_comm['COMPANY_ID'] = df_comm['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_comm['Risk_Flag'] = df_comm['CONTENT'].astype(str).str.contains('거절|불가|보류|주의').astype(int)
            comm_agg = df_comm.groupby('COMPANY_ID')['Risk_Flag'].sum().reset_index(name='NEGATIVE_COMMENT_COUNT')
            master = pd.merge(master, comm_agg, on='COMPANY_ID', how='left')

        if 'PAY_BNPL_UPLOAD_FILES.csv' in files:
            df_up = pd.read_csv(os.path.join(root, 'PAY_BNPL_UPLOAD_FILES.csv'), dtype={'COMPANY_ID': str})
            df_up['COMPANY_ID'] = df_up['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            up_agg = df_up.groupby('COMPANY_ID').size().reset_index(name='UPLOADED_FILE_COUNT')
            master = pd.merge(master, up_agg, on='COMPANY_ID', how='left')

    fill_zero_cols = ['REP_CHANGE_COUNT', 'LINKED_PARTNERS', 'NEGATIVE_COMMENT_COUNT', 'UPLOADED_FILE_COUNT']
    for c in fill_zero_cols:
        if c in master.columns:
            master[c] = master[c].fillna(0)

    output_file = "Final_Integrated_Credit_Master_v4_Schema_Mapped.csv"
    master.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n🎉 [완전체 통합 4.0 성공] 스키마 매핑된 7대 재무지표 포함 {master.shape[1]}개 심사 Feature 생성 완료!")
    
    display_cols = ['COMPANY_ID', 'LIQUIDITY_RATIO', 'DEBT_RATIO', 'CASH_RATIO', 'OPERATING_MARGIN', 'SALES_GROWTH_RATE']
    display_cols = [c for c in display_cols if c in master.columns]
    print("\n📊 [핵심 재무지표 결합 샘플 (Top 5)]")
    print(master[display_cols].head(5).to_string(index=False))
    
    return master

# 즉시 실행
df_schema_mapped = build_schema_mapped_master('.')

🏦 [Schema Mapped Engine] 스키마 명세 기반 핵심 재무 지표 통합 가동...
------------------------------------------------------------------------------------------
▶️ [1/2] 재무비율 스키마 정밀 매핑 중 (FR_VAL)...
▶️ [2/2] 대차대조표 스키마 연산 중 (현금비율 계산)...
  └─ ✅ 현금비율(CASH_RATIO) 산출 및 결합 성공!

🎉 [완전체 통합 4.0 성공] 스키마 매핑된 7대 재무지표 포함 19개 심사 Feature 생성 완료!

📊 [핵심 재무지표 결합 샘플 (Top 5)]
COMPANY_ID  LIQUIDITY_RATIO DEBT_RATIO  CASH_RATIO  OPERATING_MARGIN  SALES_GROWTH_RATE
0000000013              NaN        NaN         NaN               NaN                NaN
0000000015             52.2       자본잠식   31.196462             -45.6               56.4
0000000016              NaN        NaN         NaN               NaN                NaN
0000000017              NaN        NaN         NaN               NaN                NaN
0000000018              NaN        NaN         NaN               NaN                NaN


In [3]:
import pandas as pd
import numpy as np
import os

def integrate_income_statement(root_path='.'):
    print("📈 [Income Statement Engine] 손익계산서 수익성 지표 통합 가동...")
    print("-" * 90)
    
    # 1. 이전 단계에서 완성한 스키마 기반 마스터 테이블 로드
    master_file = "Final_Integrated_Credit_Master_v4_Schema_Mapped.csv"
    if not os.path.exists(master_file):
        print(f"❌ {master_file} 파일이 없습니다. 이전 단계를 먼저 실행해주세요.")
        return None
        
    master = pd.read_csv(master_file, dtype={'COMPANY_ID': str})
    master['COMPANY_ID'] = master['COMPANY_ID'].str.zfill(10)

    # 2. 손익계산서 파일 찾기 및 결합
    is_merged = False
    for root, dirs, files in os.walk(root_path):
        if 'COMPANY_INCOME_STATEMENT.csv' in files:
            print("▶️ 손익계산서 파일 발견! 지표 연산 및 매핑 중...")
            df_income = pd.read_csv(os.path.join(root, 'COMPANY_INCOME_STATEMENT.csv'), dtype={'COMPANY_ID': str})
            
            # 식별자 정규화 및 최신 데이터 압축
            df_income['COMPANY_ID'] = df_income['COMPANY_ID'].str.split('.').str[0].str.zfill(10)
            df_income = df_income.sort_values(by=['COMPANY_ID', 'FS_ACCT_DT']).drop_duplicates(subset=['COMPANY_ID'], keep='last')
            
            # 컬럼명 소문자 통일 (스키마 대응)
            df_income.columns = df_income.columns.str.lower()
            
            # 수익성 지표 산출
            if set(['fs_val1', 'fs_val3', 'fs_val54', 'fs_val90']).issubset(df_income.columns):
                # 0으로 나누기 방지
                df_income['fs_val1'] = df_income['fs_val1'].replace(0, np.nan)
                
                # 매출총이익률 = (매출총이익 / 매출액) * 100
                df_income['GROSS_PROFIT_MARGIN'] = (df_income['fs_val3'] / df_income['fs_val1']) * 100
                
                # 순이익률 = (당기순이익 / 매출액) * 100 (추가 지표)
                df_income['NET_PROFIT_MARGIN'] = (df_income['fs_val90'] / df_income['fs_val1']) * 100
                
                # 절대 규모 피처(Feature) 추가: 매출액(SALES) 자체도 기업 규모 판단에 중요
                df_income['SALES_REVENUE'] = df_income['fs_val1']
                
                df_income = df_income.rename(columns={'company_id': 'COMPANY_ID'})
                
                # 마스터 테이블에 결합
                cols_to_merge = ['COMPANY_ID', 'GROSS_PROFIT_MARGIN', 'NET_PROFIT_MARGIN', 'SALES_REVENUE']
                master = pd.merge(master, df_income[cols_to_merge], on='COMPANY_ID', how='left')
                is_merged = True
                print("  └─ ✅ 매출총이익률 및 수익성 지표 산출 완료!")
            else:
                print("  └─ ⚠️ 필수 컬럼(fs_val1, fs_val3 등)이 누락되어 연산 불가")

    if not is_merged:
        print("⚠️ 손익계산서 파일을 찾지 못해 결합을 건너뛰었습니다.")
        return master

    # 3. 최종 저장
    output_file = "Final_Integrated_Credit_Master_v5_Income_Added.csv"
    master.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n🎉 [완전체 통합 5.0] 손익계산서 기반 수익성 지표 추가 완료! (총 {master.shape[1]}개 Feature)")
    
    display_cols = ['COMPANY_ID', 'SALES_REVENUE', 'GROSS_PROFIT_MARGIN', 'OPERATING_MARGIN', 'NET_PROFIT_MARGIN']
    display_cols = [c for c in display_cols if c in master.columns]
    print("\n📊 [손익계산서 통합 결과 샘플 (Top 5)]")
    print(master[display_cols].head(5).to_string(index=False))
    
    return master

# 실행
df_master_v5 = integrate_income_statement('.')

📈 [Income Statement Engine] 손익계산서 수익성 지표 통합 가동...
------------------------------------------------------------------------------------------
▶️ 손익계산서 파일 발견! 지표 연산 및 매핑 중...
  └─ ✅ 매출총이익률 및 수익성 지표 산출 완료!

🎉 [완전체 통합 5.0] 손익계산서 기반 수익성 지표 추가 완료! (총 22개 Feature)

📊 [손익계산서 통합 결과 샘플 (Top 5)]
COMPANY_ID  SALES_REVENUE  GROSS_PROFIT_MARGIN  OPERATING_MARGIN  NET_PROFIT_MARGIN
0000000013            NaN                  NaN               NaN                NaN
0000000015      8079502.0           -27.161476             -45.6         -41.383541
0000000016            NaN                  NaN               NaN                NaN
0000000017            NaN                  NaN               NaN                NaN
0000000018            NaN                  NaN               NaN                NaN
